In [1]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

# ปิด warning converge ของ statsmodels (และ warning อื่น ๆ)
warnings.simplefilter("ignore", ConvergenceWarning)
warnings.filterwarnings("ignore")

import os
import sys
import json
import time
import logging
import numpy as np
import pandas as pd

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error


# =========================
# Config
# =========================
DATA_FILE = "SnP_daily_update_AMZN_features_with_target.csv"
WINDOW = 14
TARGET_COL = "y_ret_t1"

FEATURES = [
    "ma_gap_20", "vol_10", "vol_20",
    "lower_wick", "upper_wick",
    "ret_1",
    "range_pct",
    "co_ret",
    "ret_5",
]

# Orders to try (ปรับเพิ่ม/ลดได้)
ORDERS_TO_TRY = [
    (1, 0, 0),
    (2, 0, 0),
    (1, 0, 1),
    (2, 0, 1),
    (1, 0, 2),
    (2, 0, 2),
]

# ✅ Fit speed knobs (ปรับได้)
FIT_METHOD = "lbfgs"   # "lbfgs" มักเร็ว/เสถียร
MAXITER = 50           # ลดลง = เร็วขึ้น, แต่เสี่ยง converge น้อยลง
COV_TYPE = "none"      # ไม่คำนวณ covariance (ถ้าไม่ได้ใช้ std err/pvalue)

# Checkpoint / outputs
RUN_DIR = "arimax_run"
PROGRESS_JSON = os.path.join(RUN_DIR, "progress.json")
RESULTS_CSV = os.path.join(RUN_DIR, "order_results.csv")
PRED_PREFIX = os.path.join(RUN_DIR, "preds_order")
BEST_PRED_CSV = os.path.join(RUN_DIR, "best_order_predictions.csv")
BEST_MODEL_PKL = os.path.join(RUN_DIR, "best_model_fullfit.pkl")
LOG_FILE = os.path.join(RUN_DIR, "run.log")

# กันเขียนไฟล์ถี่เกิน
CHECKPOINT_EVERY_STEPS = 25


# =========================
# Logging
# =========================
def setup_logger():
    os.makedirs(RUN_DIR, exist_ok=True)
    logger = logging.getLogger("ARIMAX")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
    fh.setFormatter(fmt)
    fh.setLevel(logging.INFO)

    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    sh.setLevel(logging.INFO)

    logger.addHandler(fh)
    logger.addHandler(sh)
    return logger


# =========================
# Data
# =========================
def prepare_data(data_path, logger):
    df = pd.read_csv(data_path)

    if TARGET_COL not in df.columns:
        raise RuntimeError(f"Target column '{TARGET_COL}' not found in data")

    features = [c for c in FEATURES if c in df.columns]
    missing = [c for c in FEATURES if c not in df.columns]
    if missing:
        logger.warning(f"Missing features in data (skipped): {missing}")

    use_cols = [TARGET_COL] + features
    before = len(df)
    keep = df[use_cols].dropna().reset_index(drop=True)
    after = len(keep)

    logger.info(f"Rows before dropna={before}, after={after}, dropped={before-after}")
    logger.info(f"Using {len(features)} features: {features}")

    if len(keep) <= WINDOW:
        raise RuntimeError("Not enough rows after dropna for the given WINDOW")

    return keep, features


# =========================
# Checkpoint helpers
# =========================
def order_tag(order):
    return f"{order[0]}_{order[1]}_{order[2]}"

def pred_path_for_order(order):
    return f"{PRED_PREFIX}_{order_tag(order)}.csv"

def save_progress(obj):
    os.makedirs(RUN_DIR, exist_ok=True)
    tmp = PROGRESS_JSON + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, PROGRESS_JSON)


# =========================
# Rolling predict (with resume)
# =========================
def rolling_arimax_with_resume(df, features, window, order, logger):
    """
    ทำ rolling forecast สำหรับ order เดียว
    - ถ้าเคยมี preds file อยู่แล้ว จะอ่านมาแล้วทำต่อจากจุดสุดท้าย
    - ระหว่างทางทำ checkpoint เป็น CSV + progress.json
    - ✅ เร็วขึ้นด้วย: maxiter, method, cov_type, warm start (start_params)
    """
    n = len(df)
    max_i = (n - window)

    pred_file = pred_path_for_order(order)

    # resume
    if os.path.exists(pred_file):
        preds_df = pd.read_csv(pred_file)
        done = len(preds_df)
        start_i = done
        logger.info(f"[RESUME] order={order}, done={done}/{max_i}, continue i={start_i}")
    else:
        start_i = 0
        preds_df = pd.DataFrame(columns=["index", "y_true", "y_pred"])

    t0 = time.time()
    fail_count = 0

    # ✅ warm start params
    last_params = None

    for i in range(start_i, max_i):
        train = df.iloc[i:i+window]
        test = df.iloc[i+window]

        endog = train[TARGET_COL].to_numpy(dtype=float)

        exog = None
        exog_next = None
        if features:
            exog = train[features].to_numpy(dtype=float)
            exog_next = test[features].to_numpy(dtype=float).reshape(1, -1)

        try:
            model = SARIMAX(
                endog,
                exog=exog,
                order=order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            )

            res = model.fit(
                disp=False,
                method=FIT_METHOD,
                maxiter=MAXITER,
                cov_type=COV_TYPE,
                start_params=last_params,   # ✅ warm start
            )

            # update warm start
            last_params = res.params

            pred = res.predict(start=len(endog), end=len(endog), exog=exog_next)
            pred_val = float(pred[0])

        except Exception as e:
            fail_count += 1
            # ✅ ถ้าพัง รีเซ็ต warm start เพื่อไม่ให้พังต่อเนื่อง
            last_params = None

            if fail_count <= 5 or (fail_count % 50 == 0):
                logger.warning(f"Order={order} failed at i={i}: {repr(e)}")

            pred_val = np.nan

        row = {"index": int(i + window), "y_true": float(test[TARGET_COL]), "y_pred": pred_val}
        preds_df = pd.concat([preds_df, pd.DataFrame([row])], ignore_index=True)

        # checkpoint
        if ((i + 1) % CHECKPOINT_EVERY_STEPS == 0) or (i == max_i - 1):
            preds_df.to_csv(pred_file, index=False)

            save_progress({
                "current_order": list(order),
                "current_i_done": int(i + 1),
                "total_i": int(max_i),
                "pred_file": pred_file,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            })

            elapsed = time.time() - t0
            logger.info(f"Order={order} progress {i+1}/{max_i} | fails={fail_count} | elapsed={elapsed:.1f}s")

    return preds_df


def rmse_from_preds(preds_df):
    mask = preds_df["y_pred"].notna()
    if mask.sum() == 0:
        return np.inf, 0
    y_true = preds_df.loc[mask, "y_true"].to_numpy()
    y_pred = preds_df.loc[mask, "y_pred"].to_numpy()
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return float(rmse), int(mask.sum())


# =========================
# Fit best model on full data & save
# =========================
def fit_and_save_best_full_model(df, features, best_order, logger):
    endog_full = df[TARGET_COL].to_numpy(dtype=float)
    exog_full = df[features].to_numpy(dtype=float) if features else None

    logger.info(f"Fitting best model on FULL data with order={best_order} ...")
    model = SARIMAX(
        endog_full,
        exog=exog_full,
        order=best_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    res = model.fit(
        disp=False,
        method=FIT_METHOD,
        maxiter=max(100, MAXITER),   # full-fit ให้คอนเวิร์จนิดนึง
        cov_type=COV_TYPE,
    )

    res.save(BEST_MODEL_PKL)
    logger.info(f"Saved best full-fit model to: {BEST_MODEL_PKL}")
    return res


# =========================
# Main
# =========================
def main():
    logger = setup_logger()
    logger.info("===== ARIMAX Rolling Search Start =====")
    logger.info(f"Fit config: method={FIT_METHOD}, maxiter={MAXITER}, cov_type={COV_TYPE}")

    root = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    data_path = os.path.join(root, DATA_FILE)
    if not os.path.exists(data_path):
        logger.error(f"Data file not found: {data_path}")
        sys.exit(1)

    df, features = prepare_data(data_path, logger)
    logger.info(f"Data rows after dropna: {len(df)} | window={WINDOW}")
    logger.info(f"Orders to try: {ORDERS_TO_TRY}")

    # resume order-level
    if os.path.exists(RESULTS_CSV):
        results_df = pd.read_csv(RESULTS_CSV)
        tried = set(results_df["order"].astype(str).tolist())
        logger.info(f"[RESUME] Found existing {RESULTS_CSV} with {len(tried)} tried orders.")
    else:
        results_df = pd.DataFrame(columns=["order", "rmse", "n_pred_ok", "pred_file"])
        tried = set()

    best_rmse = np.inf
    best_order = None
    best_pred_file = None

    # set best from existing
    if len(results_df) > 0:
        tmp = results_df.copy()
        tmp["rmse"] = pd.to_numeric(tmp["rmse"], errors="coerce")
        tmp = tmp.sort_values("rmse", ascending=True)
        if tmp["rmse"].notna().any():
            best_rmse = float(tmp.iloc[0]["rmse"])
            best_order = tuple(int(x) for x in tmp.iloc[0]["order"].strip("()").split(","))
            best_pred_file = tmp.iloc[0]["pred_file"]
            logger.info(f"[RESUME] Best so far: order={best_order}, rmse={best_rmse:.6f}")

    # loop orders
    for order in ORDERS_TO_TRY:
        order_str = str(order)
        if order_str in tried:
            logger.info(f"Skip already tried order={order}")
            continue

        logger.info(f"--- Trying order={order} ---")
        preds_df = rolling_arimax_with_resume(df, features, WINDOW, order, logger)

        rmse, n_ok = rmse_from_preds(preds_df)
        pred_file = pred_path_for_order(order)

        logger.info(f"Order={order} RMSE={rmse:.6f} | ok_preds={n_ok}/{len(preds_df)} | preds={pred_file}")

        # save results immediately
        row = {"order": str(order), "rmse": rmse, "n_pred_ok": n_ok, "pred_file": pred_file}
        results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)
        results_df.to_csv(RESULTS_CSV, index=False)

        if rmse < best_rmse:
            best_rmse = rmse
            best_order = order
            best_pred_file = pred_file

        save_progress({
            "last_finished_order": list(order),
            "best_order_so_far": list(best_order) if best_order else None,
            "best_rmse_so_far": best_rmse,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        })

    if best_order is None or not np.isfinite(best_rmse):
        logger.error("No valid order produced predictions. Check data/features or reduce model complexity.")
        sys.exit(2)

    logger.info(f"===== BEST ORDER: {best_order} | RMSE={best_rmse:.6f} =====")

    # save best predictions copy
    if best_pred_file and os.path.exists(best_pred_file):
        best_preds = pd.read_csv(best_pred_file)
        best_preds.to_csv(BEST_PRED_CSV, index=False)
        logger.info(f"Saved best-order predictions to: {BEST_PRED_CSV}")

    # fit best model on full data & save
    fit_and_save_best_full_model(df, features, best_order, logger)

    logger.info("===== DONE =====")


if __name__ == "__main__":
    main()

2026-02-24 01:22:58,522 | INFO | ===== ARIMAX Rolling Search Start =====
2026-02-24 01:22:58,523 | INFO | Fit config: method=lbfgs, maxiter=50, cov_type=none
2026-02-24 01:22:58,544 | INFO | Rows before dropna=4049, after=4028, dropped=21
2026-02-24 01:22:58,545 | INFO | Using 9 features: ['ma_gap_20', 'vol_10', 'vol_20', 'lower_wick', 'upper_wick', 'ret_1', 'range_pct', 'co_ret', 'ret_5']
2026-02-24 01:22:58,545 | INFO | Data rows after dropna: 4028 | window=14
2026-02-24 01:22:58,546 | INFO | Orders to try: [(1, 0, 0), (2, 0, 0), (1, 0, 1), (2, 0, 1), (1, 0, 2), (2, 0, 2)]
2026-02-24 01:22:58,547 | INFO | --- Trying order=(1, 0, 0) ---
2026-02-24 01:22:59,747 | INFO | Order=(1, 0, 0) progress 25/4014 | fails=0 | elapsed=1.2s
2026-02-24 01:23:00,993 | INFO | Order=(1, 0, 0) progress 50/4014 | fails=0 | elapsed=2.4s
2026-02-24 01:23:02,223 | INFO | Order=(1, 0, 0) progress 75/4014 | fails=0 | elapsed=3.7s
2026-02-24 01:23:03,375 | INFO | Order=(1, 0, 0) progress 100/4014 | fails=0 | el